In [5]:
#@title 📦 Kaggle Competition Loader { display-mode: "form" }
#@markdown ---
#@markdown ### How to use
#@markdown 1. In Colab, add **two** secrets via the 🔑 Secrets panel (left sidebar):
#@markdown    - `KAGGLE_USERNAME` → your Kaggle username
#@markdown    - `KAGGLE_KEY` → your Kaggle API key
#@markdown    Toggle "Notebook access" on for both.
#@markdown 2. Run this cell (code stays hidden — double-click the title to peek).
#@markdown 3. When prompted, paste the **full Kaggle competition URL**
#@markdown    (e.g. `https://www.kaggle.com/c/titanic`) — or just the slug.
#@markdown 4. After it finishes, your data is available as:
#@markdown    - `data['train.csv']`, `data['test.csv']`, etc. → pandas DataFrames
#@markdown    - `data['some_file.ext']` → file path (non-CSV files)
#@markdown    - `data_dir` → the local folder containing everything
#@markdown ---

import os
import re
import shutil
import kagglehub
import pandas as pd


def _authenticate_kaggle():
    """
    Pulls Kaggle credentials from Colab secrets and sets them as env vars,
    so kagglehub picks them up silently instead of falling back to
    kagglehub.login()'s interactive OAuth flow (which is what was hanging
    the automated run).
    """
    try:
        from google.colab import userdata
    except ImportError:
        # Not running in Colab — assume env vars/kaggle.json are already set up.
        return

    missing = []
    for env_name in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        if os.environ.get(env_name):
            continue  # already set
        try:
            value = userdata.get(env_name)
        except Exception:
            value = None
        if value:
            os.environ[env_name] = value
        else:
            missing.append(env_name)

    if missing:
        raise RuntimeError(
            "Missing Kaggle credential(s) in Colab secrets: "
            f"{', '.join(missing)}.\n"
            "Add them via the 🔑 Secrets panel in the left sidebar "
            "(toggle 'Notebook access' on for each), then re-run this cell."
        )


def _parse_kaggle_ref(text: str):
    """
    Determines whether the input points to a Kaggle *dataset* or
    *competition*, and extracts the identifier kagglehub needs.

    Returns:
        (kind, ref) where kind is 'dataset' or 'competition', and ref is
        either "owner/dataset-slug" (datasets) or "comp-slug" (competitions).
    """
    text = text.strip().rstrip('/')

    # Full dataset URL: kaggle.com/datasets/<owner>/<slug>
    match = re.search(r'kaggle\.com/datasets/([^/?#]+)/([^/?#]+)', text)
    if match:
        return 'dataset', f"{match.group(1)}/{match.group(2)}"

    # Full competition URL: kaggle.com/c/<slug> or kaggle.com/competitions/<slug>
    match = re.search(r'kaggle\.com/(?:c|competitions)/([^/?#]+)', text)
    if match:
        return 'competition', match.group(1)

    # Any other kaggle.com URL shape — can't reliably classify it
    if "kaggle.com" in text:
        return None, text.split('/')[-1]

    # Bare input, no domain: "owner/slug" implies a dataset handle;
    # a single token is treated as a competition slug.
    if '/' in text:
        return 'dataset', text
    return 'competition', text


def fetch_kaggle_competition(comp_url: str = None, copy_to_local: bool = True):
    """
    Downloads a Kaggle dataset or competition via kagglehub and loads all CSVs.

    Args:
        comp_url: Full Kaggle dataset/competition URL, or a slug
            ("comp-slug" for a competition, "owner/dataset-slug" for a
            dataset). If None, you'll be prompted.
        copy_to_local: If True, copies files out of the kagglehub cache into
            a local folder named after the slug.

    Returns:
        data: dict mapping filename -> DataFrame (for .csv) or filepath (other files)
        working_path: directory containing the dataset files
    """
    comp_input = comp_url or input("Paste Kaggle dataset/competition URL (or slug): ").strip()
    kind, comp_id = _parse_kaggle_ref(comp_input)

    if not comp_id:
        raise ValueError("Could not parse a dataset/competition identifier from that input.")
    if kind is None:
        raise ValueError(
            f"Couldn't tell whether '{comp_input}' is a dataset or a competition. "
            "Paste the full URL (kaggle.com/datasets/... or kaggle.com/c/...) "
            "or a bare slug (owner/dataset-slug, or comp-slug)."
        )

    print(f"\n[1/3] Authenticating & downloading {kind}: '{comp_id}'...")
    _authenticate_kaggle()
    try:
        if kind == 'dataset':
            cache_path = kagglehub.dataset_download(comp_id)
        else:
            cache_path = kagglehub.competition_download(comp_id)
    except Exception as e:
        if kind == 'competition':
            raise RuntimeError(
                f"Failed to download competition '{comp_id}'. If this is a 403, "
                f"make sure you've accepted the competition rules at "
                f"https://kaggle.com/competitions/{comp_id}/rules"
            ) from e
        raise RuntimeError(
            f"Failed to download dataset '{comp_id}'. Double-check the "
            "owner/slug is correct and the dataset is public (or you have access)."
        ) from e

    local_name = comp_id.replace('/', '_')
    target_dir = os.path.join(os.getcwd(), local_name)
    if copy_to_local:
        if os.path.exists(target_dir):
            shutil.rmtree(target_dir)
        shutil.copytree(cache_path, target_dir)
        working_path = target_dir
    else:
        working_path = cache_path

    print(f"[2/3] Files localized to: {working_path}")

    data = {}
    print("\n[3/3] Inspecting dataset contents:")
    for root, _, filenames in os.walk(working_path):
        for fname in sorted(filenames):
            fpath = os.path.join(root, fname)
            if fname.endswith('.csv'):
                data[fname] = pd.read_csv(fpath)
                print(f"  • {fname:30s} shape={data[fname].shape} -> data['{fname}']")
            else:
                data[fname] = fpath
                print(f"  • {fname:30s} (path saved) -> data['{fname}']")

    return data, working_path


# ==========================================
# RUN — paste your competition URL when asked
# ==========================================
data, data_dir = fetch_kaggle_competition()
print("\nAvailable keys:", list(data.keys()))

Paste Kaggle dataset/competition URL (or slug): https://www.kaggle.com/datasets/colabsss/power-customer-energy-behavior-data

[1/3] Authenticating & downloading dataset: 'colabsss/power-customer-energy-behavior-data'...


100%|██████████| 967k/967k [00:00<00:00, 79.2MB/s]

Extracting files...
[2/3] Files localized to: /content/colabsss_power-customer-energy-behavior-data

[3/3] Inspecting dataset contents:
  • power_marketing_customer_electricity_dataset.csv shape=(15000, 38) -> data['power_marketing_customer_electricity_dataset.csv']

Available keys: ['power_marketing_customer_electricity_dataset.csv']


In [ ]:
#@title Submit predictions to Kaggle { display-mode: "form" }
#@markdown Reads Kaggle credentials from Colab secrets, validates each
#@markdown `submission_*.csv` against `sample_submission.csv`, then asks
#@markdown for a per-file y/n confirmation before submitting.
#@markdown
#@markdown ⚠️ **Most Kaggle competitions cap you at 5 submissions/day.**
#@markdown Check the "My Submissions" tab on the competition page for
#@markdown your remaining count before confirming each file — a 400/403
#@markdown error with no other obvious cause usually means you hit the cap.
#@markdown
#@markdown Assumes the Kaggle Competition Loader cell has already run
#@markdown (uses its `data` and `data_dir` globals).

from google.colab import userdata
import os, glob, pandas as pd

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_LEGACY_KEY')

COMP = os.path.basename(data_dir)   # slug taken from the loader cell, no re-entry needed
sample = data['sample_submission.csv']

for path in sorted(glob.glob("submission_*.csv")):
    sub = pd.read_csv(path)
    ok = (sub.shape == sample.shape
          and list(sub.columns) == list(sample.columns)
          and sub['id'].astype(int).equals(sample['id'].astype(int))
          and sub.isna().sum().sum() == 0)
    print(f"\n{path}: {'OK' if ok else 'MISMATCH'} — shape {sub.shape}, dtypes {dict(sub.dtypes)}")
    if not ok:
        print("  Skipping — fix the file before submitting.")
        continue
    if input(f"Submit {path}? (y/n): ").strip().lower() == 'y':
        msg = input("  Submission message: ").strip() or path
        os.system(f'kaggle competitions submit -c {COMP} -f {path} -m "{msg}"')